# Rate-Distortion Curves — Ablation Study

Compares ResSHyp variants (activation × output_padding) and all four architectures.
Data is loaded from a pre-built CSV (`SAR_DDC_FPGA_all_runs_WandB.csv`).
Run `fetch_wandb_runs.py` to regenerate the CSV from W&B.

## Dataset & Run Structure

**420 runs total** — 6 seeds × 10 λ values (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) = 60 runs per configuration.

| # | Architecture | Activation | Output padding | Notes |
|---|---|---|---|---|
| 1 | ResSHyp | GDN  | ✅ with | Ablation: output_padding effect |
| 2 | ResSHyp | GDN  | ❌ without | Ablation: output_padding effect |
| 3 | ResSHyp | ReLU | ✅ with | Ablation: activation + output_padding |
| 4 | ResSHyp | ReLU | ❌ without | **FPGA baseline** |
| 5 | SHyp    | ReLU | ❌ without | No residual blocks |
| 6 | ResFP   | ReLU | ❌ without | No hyperprior |
| 7 | FP      | ReLU | ❌ without | No residual blocks, no hyperprior |

**`model_statistics` key format:** `{architecture}_{activation}` (e.g. `ResSHyp_relu`) + `_out_pad` suffix when `no_output_padding=False`.


## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Quick ANSI colour shortcuts
r = "\033[31m"
y = "\033[33m"
g = "\033[32m"
b = "\033[34m"
e = "\033[0m"

ROOT_DIR = Path("..").resolve()
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"

# ── Shared color palette (Okabe-Ito, colorblind-safe) ────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "RD-curves"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Color preview — VS Code shows swatches for hex literals ──────────────
# Tweak these hex values here, then copy them back to notebooks/plots_colors.json.
# ── activations group ────────────────────────────────────────────────────
_c_relu = "#009E73"  # activations.relu              (green)
_c_gdn = "#CC79A7"  # activations.gdn               (mauve)
# ── output_padding group ─────────────────────────────────────────────────
_c_out_pad_with = "#0072B2"  # output_padding.with_out_pad   (blue)
_c_out_pad_without = "#D55E00"  # output_padding.no_out_pad     (orange)
# ── platforms group ──────────────────────────────────────────────────────
_c_fpga_dynamic = "#009E73"  # platforms.fpga_dynamic        (green)
_c_fpga_idle = "#85d4bc"  # platforms.fpga_idle           (pale green)
_c_gpu_dynamic = "#0072B2"  # platforms.gpu_dynamic         (blue)
_c_gpu_idle = "#7ab6d9"  # platforms.gpu_idle            (pale blue)
_c_cpu_dynamic = "#D55E00"  # platforms.cpu_dynamic         (orange)
_c_cpu_idle = "#ebb99a"  # platforms.cpu_idle            (pale orange)
# ── architectures group ──────────────────────────────────────────────────
_c_arch_resshyp = "#009E73"  # architectures.ResSHyp         (green)
_c_arch_shyp = "#E69F00"  # architectures.SHyp            (yellow-orange)
_c_arch_resfp = "#0072B2"  # architectures.ResFP           (blue)
_c_arch_fp = "#CC79A7"  # architectures.FP              (mauve)
# ── neutral / error bars ─────────────────────────────────────────────────
_c_errbar = "#555555"  # neutral dark grey for error bar caps

## Load & Filter Runs

In [ ]:
VERBOSE_FILTERING = True
print(f"{b}Loading and filtering runs... (VERBOSE={VERBOSE_FILTERING}){e}")

# ── Load raw runs from CSV (generated by fetch_wandb_runs.py) ────────────
raw_runs_df = pd.read_csv(WANDB_CSV, index_col="id")
raw_runs_df["tags"] = raw_runs_df["tags"].fillna("[]").apply(json.loads)
raw_runs_df["no_output_padding"] = raw_runs_df["no_output_padding"].map(
    {"True": True, "False": False, True: True, False: False}
)
nb_all_runs = len(raw_runs_df)
print(f"{b}Loaded {nb_all_runs} runs from {WANDB_CSV.name}{e}.")


def print_runs_info(df: pd.DataFrame, columns: list) -> None:
    """Print run_name and selected columns for each row."""
    for run_id, row in df.iterrows():
        print(f"  - {run_id:<12} {row['run_name']:<50}:", end="")
        for col in columns:
            print(f" {r}{col}={row[col]}{e},", end="")
        print()


# ── Filter: dataset ───────────────────────────────────────────────────────
ACCEPTED_DATASETS = ["TSXSSCDataModule"]
if VERBOSE_FILTERING:
    print(f"\n{b}Keeping only dataset(s): {ACCEPTED_DATASETS}{e}")
    print_runs_info(raw_runs_df[~raw_runs_df["data_name"].isin(ACCEPTED_DATASETS)], ["data_name"])
runs_df = raw_runs_df[raw_runs_df["data_name"].isin(ACCEPTED_DATASETS)].copy()

# ── Filter: seeds ─────────────────────────────────────────────────────────
ACCEPTED_SEEDS = [0, 1, 2, 3, 4, 5, 6]
if VERBOSE_FILTERING:
    print(f"\n{b}Keeping only seeds: {ACCEPTED_SEEDS}{e}")
    print_runs_info(raw_runs_df[~raw_runs_df["seed"].isin(ACCEPTED_SEEDS)], ["seed"])
runs_df = runs_df[runs_df["seed"].isin(ACCEPTED_SEEDS)]

# ── Filter: bad tags ──────────────────────────────────────────────────────
TAGS_TO_REMOVE = ["debug", "crashed", "lr_search"]
for tag in TAGS_TO_REMOVE:
    mask = runs_df["tags"].apply(lambda ts: tag in ts)
    if mask.any() and VERBOSE_FILTERING:
        print(f"\n{b}Removing {mask.sum()} runs tagged '{tag}':{e}")
        print_runs_info(runs_df[mask], ["tags"])
    runs_df = runs_df[~mask]

# ── Filter: ResFP/FP with bad learning rate (new runs are retrained at 5e-4) ─
mask_bad_lr = runs_df["architecture"].isin(["ResFP", "FP"]) & (runs_df["lr"] != 5e-4)
if mask_bad_lr.any() and VERBOSE_FILTERING:
    print(f"\n{b}Removing {mask_bad_lr.sum()} ResFP/FP runs with lr!=5e-4:{e}")
    print_runs_info(runs_df[mask_bad_lr], ["architecture", "lr"])
runs_df = runs_df[~mask_bad_lr]

# ── Filter: gdn1 activations ──────────────────────────────────────────────
mask_gdn1 = runs_df["model_name"].str.contains("gdn1", na=False)
if mask_gdn1.any() and VERBOSE_FILTERING:
    print(f"\n{b}Removing {mask_gdn1.sum()} gdn1 runs:{e}")
    print_runs_info(runs_df[mask_gdn1], ["model_name"])
runs_df = runs_df[~mask_gdn1]

# ── Summary ───────────────────────────────────────────────────────────────
nb_kept = len(runs_df)
print(
    f"\n{r}{nb_all_runs - nb_kept}{e} runs filtered out, {g}{nb_kept}{e} runs kept.\n"
    f"Runs per architecture:\n{runs_df['architecture'].value_counts().to_string()}"
)

unique_lmbda = sorted(runs_df["lmbda"].dropna().unique().astype(float).tolist())
unique_seeds = sorted(runs_df["seed"].dropna().unique().astype(int).tolist())
unique_archs = runs_df["architecture"].unique().tolist()
unique_models = runs_df["model_name"].unique().tolist()
print(
    f"\nUnique architectures ({g}{len(unique_archs)}{e}): {unique_archs} + models ({g}{len(unique_models)}{e}): {unique_models}"
    f"\nUnique λ ({g}{len(unique_lmbda)}{e}): {unique_lmbda}"
    f"\nUnique seeds ({g}{len(unique_seeds)}{e}): {unique_seeds}"
)
print(
    f"So there should be {g}{(len(unique_archs) + len(unique_models)) * len(unique_lmbda) * len(unique_seeds)}{e} runs in total."
)

## Build Statistics

In [ ]:
# ── Metrics and statistics to compute per (architecture, activation, output_padding) group ──
METRICS_OF_INTEREST = ["test/bpp"]
for metric in ["psnr", "ssim", "ms_ssim", "epd"]:
    METRICS_OF_INTEREST.append(f"test/{metric}_noisy")
    METRICS_OF_INTEREST.append(f"test/{metric}_merlin")
    METRICS_OF_INTEREST.append(f"test/{metric}_adam_noc")
STATS = ["mean", "min", "max", "std"]
STAT_COLS = [f"{m} {s}" for m in METRICS_OF_INTEREST for s in STATS]
print(
    f"Tracking {len(METRICS_OF_INTEREST)} metrics x {len(STATS)} stats = {len(STAT_COLS)} columns"
)

# ── Group by (architecture, model_name, no_output_padding) ────────────────
# NOTE: group on `architecture` (not just `model_name`) because ResSHyp and SHyp
# share the same PyTorch class name and would otherwise be merged.
# Key format: "{architecture}_{activation}[_out_pad]"  e.g. "ResSHyp_relu_out_pad"
model_statistics: dict = {}

for (arch, model_name, no_out_pad), df_group in runs_df.groupby(
    ["architecture", "model_name", "no_output_padding"]
):
    activation = model_name.split("_")[-1]  # "relu" or "gdn"
    key = f"{arch}_{activation}{'' if no_out_pad else '_out_pad'}"
    print(f"  {key:<30}  n={len(df_group)}")

    lmbda_values = sorted(df_group["lmbda"].unique().astype(float).tolist())
    stats_df = pd.DataFrame(index=lmbda_values, columns=STAT_COLS, dtype=float)

    for lmbda, df_lmbda in df_group.groupby("lmbda"):
        for metric in METRICS_OF_INTEREST:
            if metric not in df_lmbda.columns:
                continue  # metric absent from CSV (runs evaluated with older code)
            numeric = pd.to_numeric(df_lmbda[metric], errors="coerce")
            for stat in STATS:
                stats_df.loc[lmbda, f"{metric} {stat}"] = getattr(numeric, stat)()

    model_statistics[key] = stats_df

print(f"\n{b}model_statistics keys:{e} {list(model_statistics.keys())}")

### NaN / Missing-Run Diagnostic

For each group where a statistic is NaN, list the affected runs with their IDs and
which metrics are absent from the W&B summary.

Metrics can be missing in two ways:
- **(a) key absent or `None`** in the W&B summary
- **(b) key present but stored as `float NaN`** — most common cause of NaN `std` when `mean` looks valid
  (5/6 seeds have NaN → pandas `.mean()` skips them, but `.std(ddof=1)` needs ≥ 2 values)

In [ ]:
NAN_CHECK_METRICS = [
    "test/bpp",
    "test/psnr_merlin",
    "test/psnr_adam_noc",
    "test/psnr_noisy",
    "test/ssim_merlin",
    "test/epd_merlin",
]
CHECK_STAT_COLS = [
    f"{m} {s}"
    for m in ["test/bpp", "test/psnr_merlin", "test/psnr_adam_noc"]
    for s in ["mean", "std"]
]

nan_run_ids: list = []
print(f"{y}Runs with NaN / missing metrics:{e}\n")
print(f"  {'Run ID':<12} {'Name':<55} {'λ':>6}  {'seed':>4}  {'no_out_pad':>10}  Status")
print(f"  {'-' * 12} {'-' * 55} {'-' * 6}  {'-' * 4}  {'-' * 10}  ------")

for model_key, stats_df in model_statistics.items():
    # Collect λ values where any CHECK_STAT_COL is NaN
    nan_lmbdas = set()
    for col in CHECK_STAT_COLS:
        if col in stats_df.columns:
            nan_lmbdas.update(stats_df.index[stats_df[col].isna()].astype(float).tolist())
    if not nan_lmbdas:
        continue

    # Decode key → filter criteria
    has_out_pad = model_key.endswith("_out_pad")
    base = model_key[: -len("_out_pad")] if has_out_pad else model_key
    arch, activation = base.split("_")[0], base.split("_")[1]
    no_out_pad = not has_out_pad

    candidate_runs = runs_df[
        (runs_df["architecture"] == arch)
        & (runs_df["model_name"].str.endswith(f"_{activation}"))
        & (runs_df["no_output_padding"] == no_out_pad)
        & (runs_df["lmbda"].isin(nan_lmbdas))
    ]

    for run_id, run_row in candidate_runs.iterrows():
        missing = [m for m in NAN_CHECK_METRICS if m not in run_row.index or pd.isna(run_row[m])]
        nan_run_ids.append(run_id)
        flag = f"{r}MISSING/NaN{e}: {missing}" if missing else f"{g}present{e}"
        print(
            f"  {run_id:<12} {run_row['run_name']:<55} {run_row['lmbda']:>6.0f}"
            f"  {str(int(run_row['seed'])):>4}  {str(no_out_pad):>10}  {flag}"
        )
    break  # report once per (model, set of NaN lambdas)

nan_run_ids = list(dict.fromkeys(nan_run_ids))
print(f"\n{y}→ {len(nan_run_ids)} runs to re-evaluate.{e}")
if nan_run_ids:
    print("\nTo re-run update_wandb_runs.py for only these runs, add inside the main() loop:")
    print(f"    matching_runs = [r for r in matching_runs if r.id in {nan_run_ids}]")

## Plot Helpers

All plots share two composable functions:
- `plot_rd_curves(ax, …)` — draws curves on an existing axes (no labels / legend)
- `make_rd_figure(…)` — creates a standalone figure
- `make_rd_grid(…)` — creates a multi-subplot grid

Visual encoding is injected via callables so each section defines its own mapping:
- `color_fn(key)` / `linestyle_fn(key)` / `label_fn(key)` / `legend_fn(ax)` / `key_filter(key)`

In [ ]:
def plot_rd_curves(
    ax,
    statistics: dict,
    ref: str,
    color_fn,
    linestyle_fn,
    prefix: str = "test",
    key_filter=None,
    label_fn=None,
    mean_std_error_bands=True,
) -> None:
    """Draw one RD curve per key in *statistics* on *ax*. No axis labels or legend."""
    for key, stats_df in statistics.items():
        if key_filter is not None and not key_filter(key):
            continue

        bpp_col = f"{prefix}/bpp mean"
        bpp_std_col = f"{prefix}/bpp std"
        q_col = f"{prefix}/{ref} mean"
        q_min_col = f"{prefix}/{ref} min"
        q_max_col = f"{prefix}/{ref} max"
        q_mean_col = f"{prefix}/{ref} mean"
        q_std_col = f"{prefix}/{ref} std"

        if bpp_col not in stats_df.columns or q_col not in stats_df.columns:
            continue

        bpp = stats_df[bpp_col].astype(float).to_numpy()
        q = stats_df[q_col].astype(float).to_numpy()
        valid = ~(np.isnan(bpp) | np.isnan(q))
        if valid.sum() == 0:
            continue

        idx = np.argsort(bpp[valid])
        bpp_s, q_s = bpp[valid][idx], q[valid][idx]
        color = color_fn(key)
        linestyle = linestyle_fn(key)
        label = label_fn(key) if label_fn else None

        ax.plot(bpp_s, q_s, "o", linestyle=linestyle, color=color, markersize=4, label=label)

        # BPP error bars (std across seeds)
        if bpp_std_col in stats_df.columns:
            bpp_std = stats_df[bpp_std_col].astype(float).to_numpy()[valid][idx]
            if not np.all(np.isnan(bpp_std)):
                ax.errorbar(
                    bpp_s,
                    q_s,
                    xerr=bpp_std,
                    fmt="none",
                    ecolor=C["metrics"]["error_bars"],
                    capsize=2,
                    alpha=0.5,
                )

        # Min/max shaded band (across seeds)
        if mean_std_error_bands:
            if q_mean_col in stats_df.columns and q_std_col in stats_df.columns:
                q_mean = stats_df[q_mean_col].astype(float).to_numpy()[valid][idx]
                q_std = stats_df[q_std_col].astype(float).to_numpy()[valid][idx]
                if not np.all(np.isnan(q_mean)):
                    ax.fill_between(bpp_s, q_mean - q_std, q_mean + q_std, alpha=0.15, color=color)
        else:
            if q_min_col in stats_df.columns and q_max_col in stats_df.columns:
                q_min = stats_df[q_min_col].astype(float).to_numpy()[valid][idx]
                q_max = stats_df[q_max_col].astype(float).to_numpy()[valid][idx]
                if not np.all(np.isnan(q_min)):
                    ax.fill_between(bpp_s, q_min, q_max, alpha=0.15, color=color)


def make_rd_figure(
    statistics: dict,
    ref: str,
    ylabel: str,
    color_fn,
    linestyle_fn,
    key_filter=None,
    label_fn=None,
    legend_fn=None,
    title: str = "",
    save_name: str = None,
    figsize: tuple = (9, 6),
) -> None:
    """Create a standalone figure with one set of RD curves."""
    fig, ax = plt.subplots(figsize=figsize)
    plot_rd_curves(
        ax, statistics, ref, color_fn, linestyle_fn, key_filter=key_filter, label_fn=label_fn
    )
    ax.set_xlabel("Bit-rate [bpp]", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    if title:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if "ssim" in ref:
        ax.set_ylim(0.6, 1.0)
    if legend_fn:
        legend_fn(ax)
    elif label_fn:
        ax.legend(fontsize=12)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"{save_name}.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


def make_rd_grid(
    statistics: dict,
    metrics: list,
    color_fn,
    linestyle_fn,
    key_filter=None,
    label_fn=None,
    legend_fn=None,
    title: str = "",
    save_name: str = None,
    ncols: int = 3,
) -> None:
    """Create a grid of RD-curve subplots, one per metric."""
    n = len(metrics)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows), squeeze=False)

    for ax, (ref, ylabel) in zip(axes.flat, metrics):
        plot_rd_curves(
            ax, statistics, ref, color_fn, linestyle_fn, key_filter=key_filter, label_fn=label_fn
        )
        ax.set_xlabel("Bit-rate [bpp]")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.grid(True, alpha=0.3)
        if "epd" in ref:
            ax.set_ylim(0, 1.2)
        if "ssim" in ref:
            ax.set_ylim(0.6, 1.0)
        if legend_fn:
            legend_fn(ax)
        elif label_fn:
            ax.legend(fontsize=10)

    for ax in list(axes.flat)[n:]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"{save_name}.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()

## Ablation: ResSHyp Variants

Four configurations of ResSHyp: activation (ReLU / GDN) × output_padding (with / without).

- **Color** → activation (ReLU = green, GDN = mauve)
- **Linestyle** → output padding (solid = without, dotted = with)
- **Band** → min/max range across 6 seeds  |  **x-bars** → BPP std

In [ ]:
# ── Visual encoding for ablation ──────────────────────────────────────────────
# key format: "{arch}_{activation}[_out_pad]"  e.g. "ResSHyp_relu_out_pad"
# key.split("_")[1] is always "relu" or "gdn" for ResSHyp keys.


def ablation_color(key: str) -> str:
    return C["activations"][key.split("_")[1]]


def ablation_linestyle(key: str) -> str:
    return "dotted" if key.endswith("_out_pad") else "solid"


def ablation_legend(ax, fontsize: int = 12) -> None:
    handles = [
        Line2D(
            [0],
            [0],
            color=C["activations"]["relu"],
            marker="o",
            markersize=4,
            linestyle="solid",
            label="ReLU",
        ),
        Line2D(
            [0],
            [0],
            color=C["activations"]["gdn"],
            marker="o",
            markersize=4,
            linestyle="solid",
            label="GDN",
        ),
        Line2D(
            [0],
            [0],
            color=C["metrics"]["error_bars"],
            linestyle="solid",
            label="without output_padding",
        ),
        Line2D(
            [0],
            [0],
            color=C["metrics"]["error_bars"],
            linestyle="dotted",
            label="with output_padding",
        ),
    ]
    ax.legend(handles=handles, fontsize=fontsize)


def ABLATION_FILTER(key):
    return key.split("_")[0] == "ResSHyp"


ABLATION_METRICS = [
    ("psnr_merlin", "PSNR vs MERLIN [dB]"),
    ("ssim_merlin", "SSIM vs MERLIN"),
    ("epd_merlin", "EPD vs MERLIN"),
]

# ── Single PSNR figure ────────────────────────────────────────────────────────
make_rd_figure(
    model_statistics,
    ref="psnr_merlin",
    ylabel="PSNR vs MERLIN [dB]",
    color_fn=ablation_color,
    linestyle_fn=ablation_linestyle,
    key_filter=ABLATION_FILTER,
    legend_fn=ablation_legend,
    save_name="RD-curves_ablation_PSNR-MERLIN",
)

In [ ]:
# ── 3-metric grid (PSNR / SSIM / EPD vs MERLIN) ──────────────────────────────
make_rd_grid(
    model_statistics,
    metrics=ABLATION_METRICS,
    color_fn=ablation_color,
    linestyle_fn=ablation_linestyle,
    key_filter=ABLATION_FILTER,
    legend_fn=ablation_legend,
    title="ResSHyp ablation — compression performance vs MERLIN reference",
    save_name="RD-curves_ablation_subplots",
)

## Architecture Comparison

All four architectures (**ResSHyp**, **SHyp**, **ResFP**, **FP**) on the ReLU + no output_padding baseline.

- **Color** → architecture (Okabe-Ito palette)
- **Band** → min/max range across seeds  |  **x-bars** → BPP std

In [ ]:
# ── Derive arch_statistics from model_statistics ─────────────────────────────
# The common baseline for all architectures is ReLU + no_output_padding.
# "ResSHyp_relu" (no _out_pad suffix) is exactly that entry in model_statistics.
ARCH_ORDER = ["ResSHyp", "SHyp", "ResFP", "FP"]
ARCH_COLORS = {arch: C["architectures"][arch] for arch in ARCH_ORDER if arch in C["architectures"]}

arch_statistics = {
    arch: model_statistics[f"{arch}_relu"]
    for arch in ARCH_ORDER
    if f"{arch}_relu" in model_statistics
}
print(f"Architectures in baseline: {list(arch_statistics.keys())}")


# ── Visual encoding ───────────────────────────────────────────────────────────
def arch_color(key: str) -> str:
    return ARCH_COLORS.get(key, "#999999")


def arch_linestyle(key: str) -> str:
    return "solid"


def arch_label(key: str) -> str:
    return key

In [ ]:
# ── One figure per metric ─────────────────────────────────────────────────────
ARCH_METRICS = [
    ("psnr_merlin", "PSNR vs MERLIN [dB]"),
    ("ssim_merlin", "SSIM vs MERLIN"),
    ("epd_merlin", "EPD vs MERLIN"),
]

for ref, ylabel in ARCH_METRICS:
    make_rd_figure(
        arch_statistics,
        ref=ref,
        ylabel=ylabel,
        color_fn=arch_color,
        linestyle_fn=arch_linestyle,
        label_fn=arch_label,
        title=f"Architecture comparison — {ylabel} (ReLU, no output_padding)",
        save_name=f"RD-curves_arch_comparison_{ref.replace('/', '_')}",
    )